# 泊松分布完整教程：从稀有事件到概率计算

## 📚 教学目标
1. 理解泊松分布的适用条件（稀有事件）
2. 掌握泊松概率公式：$P(x) = \frac{\mu^x \times e^{-\mu}}{x!}$
3. 用 scipy.stats.poisson 计算概率
4. 理解泊松分布与二项分布的关系
5. 应用泊松分布求解实际问题（飓风发生、缺陷率）

**参考书**：《基础统计学(第14版)》（Triola）第 5-3 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

## 1. 场景设定：泊松分布

### 🎯 问题
生活中有很多"计数型"事件：
- 一天内某网站用户登录的次数
- 一年中大西洋飓风发生的次数
- 每页的拼写错误数量
- 1 小时内到达急诊室的病人数量
- 每天发生的交通事故数量

这些事件的共同特点是：在**固定的区间**内，某个事件**随机且独立**地发生。这类问题就适合用**泊松分布**来建模。

### 📖 书中的定义
> 泊松分布是适合描述特定区间内某一事件发生次数的离散概率分布。
> 随机变量 $x$ 是区间内事件发生的次数。区间可以是时间、距离、面积、体积或其他类似的单位。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.special import factorial

# 设置中文字体和样式
import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

np.random.seed(42)
print("✅ 库导入完成")

## 2. 泊松分布的四个条件

### 📐 条件检查
一个过程是否服从泊松分布，需要检查四个条件：

| 条件 | 说明 | 检查要点 |
|------|------|----------|
| 1. 区间内计数 | $x$ 是某一区间内事件发生的次数 | 时间/距离/面积/体积 |
| 2. 随机性 | 事件的发生必须是随机的 | 不能有系统性模式 |
| 3. 独立性 | 事件的发生必须是相互独立的 | 一个事件不影响另一个 |
| 4. 均匀性 | 事件的发生在区间内必须呈均匀分布 | 不会出现集中爆发 |

### 📖 书中的例子
> 大西洋飓风的发生：
> (1) 每年飓风次数 ✓ (2) 随机发生 ✓ (3) 一次飓风不影响另一次 ✓ (4) 全年均匀分布 ✓

### 💡 与二项分布的区别
- **二项分布**：固定试验次数 $n$，每次试验成功/失败
- **泊松分布**：没有固定试验次数，只关心区间内事件发生的次数

## 3. 泊松概率公式

### 📐 计算公式
在泊松分布中，区间内事件发生 $x$ 次的概率：
$$P(x) = \frac{\mu^x \times e^{-\mu}}{x!}$$

其中：
- $x$ = 区间内事件发生的次数（$x = 0, 1, 2, \ldots$）
- $\mu$ = 区间内事件发生次数的均值（即期望值）
- $e$ = 自然常数 $\approx 2.71828$
- $x!$ = $x$ 的阶乘

### 📐 泊松分布的参数
- 均值：$\mu$（即参数本身）
- 标准差：$\sigma = \sqrt{\mu}$

### 💡 关键洞察
泊松分布只有一个参数 $\mu$！一旦知道了均值，整个分布就完全确定了。

In [ ]:
# ========== 步骤 1: 手算泊松概率 ==========

# 场景：某医院急诊室平均每小时到达 4 位病人
# 求：下一个小时内恰好到达 2 位病人的概率

mu = 4  # 每小时平均到达人数
x = 2   # 我们想知道恰好 2 人到达的概率

# 手算 P(x=2)
# 因子1: mu^x
mu_power = mu ** x
# 因子2: e^(-mu)
e_neg_mu = np.exp(-mu)
# 因子3: x!
x_factorial = factorial(x, exact=True)

# 手算概率
prob_manual = (mu_power * e_neg_mu) / x_factorial

print(f"\n📊 步骤 1: 手算泊松概率 P(x={x})")
print(f"  参数: μ = {mu}（每小时平均到达 {mu} 人）")
print(f"  计算公式: P(x) = (μ^x × e^(-μ)) / x!")
print(f"\n  计算过程:")
print(f"    因子1: μ^{x} = {mu}^{x} = {mu_power}")
print(f"    因子2: e^(-μ) = e^(-{mu}) = {e_neg_mu:.6f}")
print(f"    因子3: {x}! = {x_factorial}")
print(f"\n  P({x}) = ({mu_power} × {e_neg_mu:.6f}) / {x_factorial}")
print(f"  P({x}) = {prob_manual:.6f}")
print(f"\n💡 下一个小时内恰好到达 {x} 位病人的概率 ≈ {prob_manual:.4f}（约 {prob_manual*100:.2f}%）")

In [ ]:
# ========== 步骤 2: 用 scipy.stats.poisson 验证 ==========

# 使用 scipy 计算
prob_scipy = stats.poisson.pmf(x, mu)

print(f"\n📊 步骤 2: 用 scipy.stats.poisson 验证")
print(f"  手算 P({x}) = {prob_manual:.6f}")
print(f"  scipy P({x}) = {prob_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

## 4. 完整的泊松概率分布

### 📐 概念
我们可以计算 $x = 0, 1, 2, \ldots$ 的所有概率，得到完整的泊松概率分布。

### 📖 书中的例子
> 当 $\mu = 4$ 时，我们可以计算 $P(0), P(1), P(2), \ldots$ 并绘制分布图。

In [ ]:
# ========== 步骤 3: 完整的泊松概率分布 ==========

# 计算 x = 0 到 15 的所有概率
x_values = np.arange(0, 16)
probs = stats.poisson.pmf(x_values, mu)

# 创建分布表
dist_table = pd.DataFrame({
    'x': x_values,
    'P(x)': probs,
    'P(x) %': probs * 100
})

print(f"\n📊 步骤 3: μ = {mu} 的完整泊松概率分布")
print(f"  {'x':<5} {'P(x)':<12} {'P(x) %':<10}")
print(f"  {'-'*30}")
for _, row in dist_table.iterrows():
    print(f"  {int(row['x']):<5} {row['P(x)']:<12.6f} {row['P(x) %']:<10.2f}%")

print(f"\n💡 验证: 所有概率之和 = {probs.sum():.6f}（应为 1.000000）")

In [ ]:
# ========== 可视化: 泊松概率分布 ==========

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(x_values, probs, color='steelblue', alpha=0.7, edgecolor='white')
ax.set_xlabel('Number of Events (x)', fontsize=12)
ax.set_ylabel('Probability P(x)', fontsize=12)
ax.set_title(f'Poisson Distribution (μ = {mu})', fontsize=14, fontweight='bold')
ax.set_xticks(x_values)
ax.grid(axis='y', alpha=0.3)

# 标记均值位置
ax.axvline(x=mu, color='#e74c3c', linestyle='--', alpha=0.7, label=f'Mean = {mu}')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  图中展示了 μ = {mu} 的泊松概率分布。")
print(f"  红色虚线标记了均值 μ = {mu} 的位置。")
print(f"  分布呈右偏形态，随着 x 增大，概率逐渐减小。")

## 5. 累积概率与实际应用

### 📐 累积概率
在实际问题中，我们经常需要计算"不超过某值"或"至少某值"的概率：

$$P(x \leq k) = \sum_{i=0}^{k} P(i)$$

$$P(x \geq k) = 1 - P(x \leq k-1)$$

In [ ]:
# ========== 步骤 4: 累积概率计算 ==========

# 场景：急诊室平均每小时到达 4 位病人
# 求：(1) 不超过 3 人的概率  (2) 至少 6 人的概率

k1 = 3  # 不超过 3 人
k2 = 6  # 至少 6 人

# (1) P(x <= 3)
prob_le_3 = stats.poisson.cdf(k1, mu)

# (2) P(x >= 6) = 1 - P(x <= 5)
prob_ge_6 = 1 - stats.poisson.cdf(k2 - 1, mu)

print(f"\n📊 步骤 4: 累积概率计算")
print(f"  参数: μ = {mu}（每小时平均到达 {mu} 人）")
print(f"\n  (1) P(x ≤ {k1}) = {prob_le_3:.6f}（约 {prob_le_3*100:.2f}%）")
print(f"      → 下一小时内不超过 {k1} 位病人的概率")
print(f"\n  (2) P(x ≥ {k2}) = 1 - P(x ≤ {k2-1}) = 1 - {stats.poisson.cdf(k2-1, mu):.6f}")
print(f"      P(x ≥ {k2}) = {prob_ge_6:.6f}（约 {prob_ge_6*100:.2f}%）")
print(f"      → 下一小时内至少 {k2} 位病人的概率")

# 用求和验证
prob_le_3_sum = sum(stats.poisson.pmf(i, mu) for i in range(k1 + 1))
print(f"\n🔬 验证: 用求和计算 P(x ≤ {k1}) = {prob_le_3_sum:.6f}")
print(f"  ✅ 验证通过！")

## 6. 泊松分布的参数特征

### 📐 参数与分布形态
泊松分布只有一个参数 $\mu$，它决定了分布的形态：
- $\mu$ 越小，分布越右偏
- $\mu$ 越大，分布越接近对称（类似正态分布）

### 📐 标准差
$$\sigma = \sqrt{\mu}$$

### 💡 关键洞察
泊松分布的方差等于均值！这是泊松分布的一个重要特征。

In [ ]:
# ========== 步骤 5: 不同 μ 值的泊松分布对比 ==========

mu_values = [1, 2, 4, 8, 16]
x_max = 30
x_range = np.arange(0, x_max + 1)

fig, axes = plt.subplots(1, len(mu_values), figsize=(20, 4), sharey=True)

for i, mu_val in enumerate(mu_values):
    probs_val = stats.poisson.pmf(x_range, mu_val)
    axes[i].bar(x_range, probs_val, color='steelblue', alpha=0.7, edgecolor='white')
    axes[i].axvline(x=mu_val, color='#e74c3c', linestyle='--', alpha=0.7)
    axes[i].set_title(f'μ = {mu_val}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('x', fontsize=10)
    if i == 0:
        axes[i].set_ylabel('P(x)', fontsize=10)
    axes[i].grid(axis='y', alpha=0.3)
    # 设置 x 轴范围
    axes[i].set_xlim(-0.5, min(mu_val * 3 + 2, x_max) + 0.5)

plt.suptitle('Poisson Distributions with Different μ Values', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  展示了不同 μ 值的泊松分布。")
print(f"  红色虚线标记了均值 μ 的位置。")
print(f"  观察：μ 越小，分布越右偏；μ 越大，分布越接近对称。")
print(f"  当 μ ≥ 10 时，泊松分布可以用正态分布来近似。")

## 7. 大样本应用：飓风发生次数

### 📖 书中的例子
> 大西洋飓风的发生服从泊松分布。假设每年平均发生 6 次飓风。
> 求：(1) 恰好发生 8 次的概率  (2) 不超过 3 次的概率  (3) 至少 10 次的概率

### 📐 计算公式
$$P(x) = \frac{\mu^x \times e^{-\mu}}{x!}$$

In [ ]:
# ========== 步骤 6: 飓风发生次数的泊松分布 ==========

# 参数
mu_hurricane = 6  # 每年平均发生 6 次飓风

# 计算各种概率
prob_exactly_8 = stats.poisson.pmf(8, mu_hurricane)
prob_le_3 = stats.poisson.cdf(3, mu_hurricane)
prob_ge_10 = 1 - stats.poisson.cdf(9, mu_hurricane)

print(f"\n📊 步骤 6: 飓风发生次数的泊松分布")
print(f"  参数: μ = {mu_hurricane}（每年平均发生 {mu_hurricane} 次飓风）")
print(f"  标准差: σ = √μ = √{mu_hurricane} = {np.sqrt(mu_hurricane):.4f}")
print(f"\n  (1) P(x = 8) = {prob_exactly_8:.6f}（约 {prob_exactly_8*100:.2f}%）")
print(f"      → 恰好发生 8 次飓风的概率")
print(f"\n  (2) P(x ≤ 3) = {prob_le_3:.6f}（约 {prob_le_3*100:.2f}%）")
print(f"      → 不超过 3 次飓风的概率")
print(f"\n  (3) P(x ≥ 10) = {prob_ge_10:.6f}（约 {prob_ge_10*100:.2f}%）")
print(f"      → 至少 10 次飓风的概率")

In [ ]:
# ========== 可视化: 飓风发生次数分布 ==========

x_hurricane = np.arange(0, 20)
probs_hurricane = stats.poisson.pmf(x_hurricane, mu_hurricane)

fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(x_hurricane, probs_hurricane, color='steelblue', alpha=0.7, edgecolor='white')

# 标记特殊区域
ax.bar(x_hurricane[x_hurricane <= 3], probs_hurricane[x_hurricane <= 3], 
       color='#2ecc71', alpha=0.7, edgecolor='white', label='P(x ≤ 3)')
ax.bar(x_hurricane[x_hurricane >= 10], probs_hurricane[x_hurricane >= 10], 
       color='#e74c3c', alpha=0.7, edgecolor='white', label='P(x ≥ 10)')

ax.axvline(x=mu_hurricane, color='#e67e22', linestyle='--', alpha=0.7, label=f'Mean = {mu_hurricane}')

ax.set_xlabel('Number of Hurricanes (x)', fontsize=12)
ax.set_ylabel('Probability P(x)', fontsize=12)
ax.set_title(f'Hurricane Frequency Distribution (μ = {mu_hurricane})', fontsize=14, fontweight='bold')
ax.set_xticks(x_hurricane)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  绿色区域：P(x ≤ 3) = {prob_le_3:.4f}，即飓风次数不超过 3 次的概率。")
print(f"  红色区域：P(x ≥ 10) = {prob_ge_10:.4f}，即飓风次数至少 10 次的概率。")
print(f"  橙色虚线：均值 μ = {mu_hurricane}。")

## 8. 泊松分布与二项分布的关系

### 📐 近似关系
当二项分布的 $n$ 很大、$p$ 很小时，二项分布可以用泊松分布来近似：

$$\mu = n \times p$$

### 💡 经验法则
- 当 $n \geq 100$ 且 $np \leq 10$ 时，泊松近似效果很好
- 泊松分布计算更简单（只有一个参数）

### 📖 书中的例子
> 假设某工厂生产的零件缺陷率为 0.02（2%）。随机检查 100 个零件，
> 求恰好有 3 个缺陷品的概率。
> 用二项分布：$n=100, p=0.02$，$\mu = np = 2$

In [ ]:
# ========== 步骤 7: 泊松近似二项分布 ==========

# 场景：缺陷率 p=0.02，检查 n=100 个零件
n_binom = 100
p_binom = 0.02
mu_approx = n_binom * p_binom  # μ = np = 2

x_compare = np.arange(0, 11)

# 二项分布概率
probs_binom = stats.binom.pmf(x_compare, n_binom, p_binom)
# 泊松近似概率
probs_poisson = stats.poisson.pmf(x_compare, mu_approx)

# 创建对比表
compare_table = pd.DataFrame({
    'x': x_compare,
    'Binomial': probs_binom,
    'Poisson': probs_poisson,
    'Difference': np.abs(probs_binom - probs_poisson)
})

print(f"\n📊 步骤 7: 泊松近似二项分布")
print(f"  二项分布参数: n = {n_binom}, p = {p_binom}")
print(f"  泊松近似参数: μ = np = {mu_approx}")
print(f"\n  {'x':<5} {'Binomial':<12} {'Poisson':<12} {'Diff':<10}")
print(f"  {'-'*40}")
for _, row in compare_table.iterrows():
    print(f"  {int(row['x']):<5} {row['Binomial']:<12.6f} {row['Poisson']:<12.6f} {row['Difference']:<10.6f}")

print(f"\n💡 观察：泊松近似与二项分布的结果非常接近！")
print(f"  最大差异: {compare_table['Difference'].max():.6f}")

In [ ]:
# ========== 可视化: 二项分布 vs 泊松近似 ==========

fig, ax = plt.subplots(figsize=(12, 6))

width = 0.35
x_pos = x_compare - width/2

ax.bar(x_pos, probs_binom, width, color='steelblue', alpha=0.7, edgecolor='white', label='Binomial (n=100, p=0.02)')
ax.bar(x_compare + width/2, probs_poisson, width, color='#e74c3c', alpha=0.7, edgecolor='white', label=f'Poisson (μ={mu_approx})')

ax.set_xlabel('Number of Defects (x)', fontsize=12)
ax.set_ylabel('Probability P(x)', fontsize=12)
ax.set_title('Binomial vs Poisson Approximation', fontsize=14, fontweight='bold')
ax.set_xticks(x_compare)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  蓝色：二项分布 (n=100, p=0.02) 的精确概率。")
print(f"  红色：泊松分布 (μ=2) 的近似概率。")
print(f"  两者非常接近，验证了泊松分布是二项分布的良好近似。")

## 9. 模拟验证

### 📐 模拟实验
通过模拟实验，我们可以验证泊松分布的理论概率。

### 📖 实验设计
> 模拟 10000 个小时的急诊室病人到达情况（$\mu = 4$），
> 统计每小时到达人数的分布，并与理论值比较。

In [ ]:
# ========== 步骤 8: 模拟验证 ==========

# 模拟参数
n_simulations = 10000
mu_sim = 4

# 生成模拟数据
simulated_data = stats.poisson.rvs(mu_sim, size=n_simulations)

# 统计模拟结果
x_sim, counts_sim = np.unique(simulated_data, return_counts=True)
probs_simulated = counts_sim / n_simulations

# 理论概率
x_theory = np.arange(0, max(x_sim) + 1)
probs_theory = stats.poisson.pmf(x_theory, mu_sim)

print(f"\n📊 步骤 8: 模拟验证")
print(f"  模拟次数: {n_simulations}")
print(f"  参数: μ = {mu_sim}")
print(f"\n  {'x':<5} {'Simulated':<12} {'Theoretical':<12} {'Diff':<10}")
print(f"  {'-'*40}")
for i in range(min(len(x_sim), len(x_theory))):
    if i < len(probs_simulated):
        diff = abs(probs_simulated[i] - probs_theory[i])
        print(f"  {x_sim[i]:<5} {probs_simulated[i]:<12.6f} {probs_theory[i]:<12.6f} {diff:<10.6f}")

print(f"\n💡 模拟结果与理论值非常接近，验证了泊松分布的正确性。")

In [ ]:
# ========== 可视化: 模拟 vs 理论 ==========

fig, ax = plt.subplots(figsize=(12, 6))

width = 0.35
x_pos_sim = x_sim[:len(probs_simulated)] - width/2
x_pos_theory = x_theory[:len(probs_theory)] + width/2

ax.bar(x_pos_sim, probs_simulated, width, color='steelblue', alpha=0.7, edgecolor='white', label='Simulated')
ax.bar(x_pos_theory, probs_theory, width, color='#e74c3c', alpha=0.7, edgecolor='white', label='Theoretical')

ax.set_xlabel('Number of Arrivals (x)', fontsize=12)
ax.set_ylabel('Probability P(x)', fontsize=12)
ax.set_title(f'Simulated vs Theoretical Poisson Distribution (μ = {mu_sim})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  蓝色：模拟实验的频率分布。")
print(f"  红色：泊松分布的理论概率。")
print(f"  随着模拟次数增加，模拟结果会越来越接近理论值。")

## 10. 汇总报告

### 📊 结果汇总

In [ ]:
# ========== 汇总报告 ==========

print("=" * 60)
print("📋 泊松分布学习汇总")
print("=" * 60)

print(f"\n🎯 核心公式:")
print(f"   P(x) = (μ^x × e^(-μ)) / x!")

print(f"\n📊 参数特征:")
print(f"   均值: μ（唯一参数）")
print(f"   标准差: σ = √μ")
print(f"   方差: σ² = μ")

print(f"\n🔬 适用条件:")
print(f"   1. 区间内计数（时间/距离/面积/体积）")
print(f"   2. 事件随机发生")
print(f"   3. 事件相互独立")
print(f"   4. 事件在区间内均匀分布")

print(f"\n🔗 与二项分布的关系:")
print(f"   当 n 很大、p 很小时，二项分布 → 泊松分布")
print(f"   μ = n × p")

print(f"\n📝 常见应用场景:")
print(f"   • 飓风/地震等自然灾害发生次数")
print(f"   • 急诊室/客服中心到达人数")
print(f"   • 产品缺陷率")
print(f"   • 网站访问量")
print(f"   • 放射性衰变次数")

print("\n" + "=" * 60)

## 📌 核心概念回顾

### 📌 泊松分布 (Poisson Distribution)
- **定义**: 描述特定区间内某一事件发生次数的离散概率分布
- **公式**: $P(x) = \frac{\mu^x \times e^{-\mu}}{x!}$
- **参数**: 唯一参数 $\mu$（均值），标准差 $\sigma = \sqrt{\mu}$
- **适用条件**: 区间内计数、随机性、独立性、均匀性

### 📌 泊松分布 vs 二项分布
- **二项分布**: 固定试验次数 $n$，每次试验成功/失败，参数 $n, p$
- **泊松分布**: 没有固定试验次数，只关心区间内事件次数，参数 $\mu$
- **近似关系**: 当 $n$ 很大、$p$ 很小时，二项分布可用泊松分布近似，$\mu = np$

### 📌 累积概率
- $P(x \leq k) = \sum_{i=0}^{k} P(i)$：不超过 $k$ 次的概率
- $P(x \geq k) = 1 - P(x \leq k-1)$：至少 $k$ 次的概率

### 🔗 完整流程
```
确定问题 → 检查条件 → 确定参数 μ → 计算概率 → 解读结果
    ↓           ↓           ↓           ↓           ↓
  计数型     四个条件     区间均值     公式/scipy    实际意义
```

## ❌ 常见误区

### ❌ 误区 1: 把泊松分布当二项分布用
**✓ 正确理解**: 如果问题没有固定的试验次数 $n$，而是问"某个区间内发生多少次"，应该用泊松分布，而不是二项分布。

### ❌ 误区 2: 忘记检查四个条件
**✓ 正确做法**: 使用泊松分布前，必须检查：(1) 区间内计数 (2) 随机性 (3) 独立性 (4) 均匀性。如果条件不满足，泊松分布可能不适用。

### ❌ 误区 3: 混淆均值和标准差
**✓ 正确理解**: 泊松分布的均值是 $\mu$，标准差是 $\sigma = \sqrt{\mu}$（不是 $\mu$ 本身）。

### ❌ 误区 4: 忘记泊松分布是离散分布
**✓ 正确理解**: 泊松分布的 $x$ 只能取非负整数（0, 1, 2, ...），不能取小数或负数。

### ❌ 误区 5: 在 $n$ 小 $p$ 大时用泊松近似二项分布
**✓ 正确理解**: 泊松近似适用于 $n$ 很大（$\geq 100$）且 $p$ 很小（$\leq 0.05$）的情况。当 $p$ 较大时，近似效果很差。